# 03 — Rotulagem automatizada baseada em regras

Integração dos focos de queimadas com o RONI da NOAA e rotulagem por ciclo, fase do ENOS e foco crítico.

## 1. Leitura dos dados

In [1]:
from importlib.util import find_spec
from pathlib import Path

import pandas as pd

if find_spec("google") is not None and find_spec("google.colab") is not None:
    pasta_dados = Path("/content/PROJETO-COLETA-E-RASPAGEM-DE-DADOS/dados")
    if not pasta_dados.exists():
        from google.colab import drive

        drive.mount("/content/drive")
        pasta_dados = Path("/content/drive/MyDrive/PROJETO-COLETA-E-RASPAGEM-DE-DADOS/dados")
else:
    pasta_dados = Path("../dados")

df = pd.read_parquet(pasta_dados / "queimadas_limpo.parquet")
dados_roni = pd.read_csv(pasta_dados / "roni.csv")

## 2. Integração por mês central do trimestre

A última observação do `roni.csv` é `ASO 2026`, com mês central em setembro. Os focos de outubro e novembro de 2026 ficam com `trimestre` e `roni` ausentes.

In [2]:
mes_central = {
    "DJF": "01",
    "JFM": "02",
    "FMA": "03",
    "MAM": "04",
    "AMJ": "05",
    "MJJ": "06",
    "JJA": "07",
    "JAS": "08",
    "ASO": "09",
    "SON": "10",
    "OND": "11",
    "NDJ": "12",
}
dados_roni["ano_mes"] = dados_roni["ano"].astype(str) + "-" + dados_roni["trimestre"].map(mes_central)

df = pd.merge(df, dados_roni[["ano_mes", "trimestre", "roni"]], on="ano_mes", how="left")

## 3. Rotulagem por ciclo

In [3]:
sem_el_nino = df["ano_mes"].between("2024-12", "2025-11")
com_el_nino = df["ano_mes"].between("2025-12", "2026-11")

df.loc[sem_el_nino, "ciclo"] = "2024-2025 (Sem El Niño)"
df.loc[com_el_nino, "ciclo"] = "2025-2026 (El Niño)"

## 4. Rotulagem da fase do ENOS

In [4]:
df["fase_enso"] = "Neutro"
df.loc[df["roni"] >= 0.5, "fase_enso"] = "El Niño"
df.loc[df["roni"] <= -0.5, "fase_enso"] = "La Niña"
df.loc[df["roni"].isna(), "fase_enso"] = "Sem dado RONI"

## 5. Rotulagem de foco crítico

In [5]:
df["foco_critico"] = ((df["RiscoFogo"] >= 0.8) & (df["DiaSemChuva"] >= 15)).astype(int)

## 6. Contagens dos rótulos

In [6]:
display(df["ciclo"].value_counts())
display(df["fase_enso"].value_counts())
display(df["foco_critico"].value_counts())

ciclo
2024-2025 (Sem El Niño)    853402
2025-2026 (El Niño)        780213
Name: count, dtype: int64

fase_enso
La Niña          808507
El Niño          373736
Sem dado RONI    293864
Neutro           157508
Name: count, dtype: int64

foco_critico
0    1145676
1     487939
Name: count, dtype: int64

## 7. Exportação da base rotulada

In [7]:
df.to_parquet(pasta_dados / "queimadas_rotulado.parquet", index=False)